# Train the PCB check: "is there a PCB in this picture?"

**Project:** Bharat AI Hackathon, on-device PCB inspection on Raspberry Pi 5.

**Why:** the station must give a third answer, *no PCB in view*, instead of PASS or FAIL when the camera
is not looking at a board. This notebook trains a very small image classifier (YOLOv8n-cls) with two
classes, `pcb` and `not_pcb`, and gives you `pcb_gate.onnx` for the Pi.

**Pictures it learns from (all downloaded or generated, nothing to collect):**
- `pcb`: the 693 photos of the PKU PCB dataset, plus **board-on-a-table composites**: the same PCB photos shrunk,
  tilted and placed on random backgrounds, because at the station a board fills only part of the picture.
- `not_pcb`: 1,000 everyday photos from Imagenette (a free subset of ImageNet), plus **plain scenes** generated
  here: blank paper, plain tables, gradients, shadows and printed-text-like lines, which is what an empty
  inspection stand looks like.

**Optional, for later:** your own photos from the station camera make the check much more reliable. To use them,
put them in Google Drive as `pcb_gate_photos/pcb` and `pcb_gate_photos/not_pcb` and set `USE_OWN_PHOTOS = True`
in Step 3. They are then scored separately at the end.

**Before you start:** in Colab choose **Runtime → Change runtime type → T4 GPU → Save**.
Run the cells from top to bottom with **Shift + Enter**. The whole notebook takes roughly 15 to 25 minutes.

## Step 1: Check that the GPU is on
This should print a table that mentions **Tesla T4** (or another GPU).

In [ ]:
!nvidia-smi

## Step 2: Install the libraries
- `ultralytics` contains YOLO.
- `kagglehub` downloads the PCB dataset from Kaggle.

In [ ]:
!pip install -q ultralytics kagglehub

## Step 3: Your own photos (optional, off by default)
Leave `USE_OWN_PHOTOS = False` to train on downloaded and generated pictures only.
Set it to `True` once you have photos from the station camera in Google Drive (see the top of the notebook).

In [ ]:
import glob, os, shutil

USE_OWN_PHOTOS = False
DRIVE_FOLDER = "/content/drive/MyDrive/pcb_gate_photos"
IMAGE_TYPES = (".jpg", ".jpeg", ".png", ".bmp")

own = {"pcb": [], "not_pcb": []}
if USE_OWN_PHOTOS:
    from google.colab import drive
    drive.mount("/content/drive")
    for label in own:
        folder = os.path.join(DRIVE_FOLDER, label)
        own[label] = sorted(p for p in glob.glob(os.path.join(folder, "**", "*"), recursive=True)
                            if p.lower().endswith(IMAGE_TYPES))
        print(f"Your '{label}' photos: {len(own[label])}")
    for label, paths in own.items():
        if len(paths) < 10:
            raise SystemExit(f"Please add at least 10 photos to {DRIVE_FOLDER}/{label}, or set USE_OWN_PHOTOS = False.")
else:
    print("Not using your own photos (USE_OWN_PHOTOS = False).")

## Step 4: Download the PCB dataset and the everyday photos
- The PCB dataset is the same one used for the defect detector (about 2 GB).
- Imagenette is a free set of everyday photos (fish, dogs, churches, trucks and so on), about 330 MB.

In [ ]:
import random, tarfile, urllib.request
import kagglehub

pcb_dir = kagglehub.dataset_download("akhatova/pcb-defects")
dataset_pcb = sorted(p for p in glob.glob(os.path.join(pcb_dir, "**", "*.jpg"), recursive=True)
                     if "rotation" not in p.lower())
print("PCB dataset photos:", len(dataset_pcb))

IMAGENETTE_URL = "https://s3.amazonaws.com/fast-ai-imageclas/imagenette2-320.tgz"
if not os.path.isdir("/content/imagenette2-320"):
    urllib.request.urlretrieve(IMAGENETTE_URL, "/content/imagenette.tgz")
    with tarfile.open("/content/imagenette.tgz") as tar:
        tar.extractall("/content")
everyday = sorted(glob.glob("/content/imagenette2-320/**/*.JPEG", recursive=True))
everyday = random.Random(0).sample(everyday, min(1000, len(everyday)))
print("Everyday photos used:", len(everyday))

## Step 5: Build the training folders
What this cell does:
1. **Makes extra pictures** that look more like the inspection station:
   - *board-on-a-table composites* (`pcb`): a PCB photo shrunk to 35 to 90% of the picture, slightly tilted,
     on a random background (an everyday photo or a plain surface), sometimes with a shadow or a colour cast;
   - *plain scenes* (`not_pcb`): blank or lined paper, plain or wood-like surfaces, gradients and shadows.
2. **Splits all pictures** into `train` (to learn from) and `val` (to test on): 15% is kept for testing and never
   used for learning. Your own photos, if used, are split 75 / 25.
3. **Shrinks every picture** to at most 640 pixels on its longest side, to make training faster. The model
   itself looks at 224 x 224 pixels.
4. **Repeats your own training photos three times** (if used), because there are few of them and they matter most.

YOLO's classifier expects one folder per class: `train/pcb`, `train/not_pcb`, `val/pcb`, `val/not_pcb`.

In [ ]:
import numpy as np
import cv2
from PIL import Image, ImageOps

DATA = "/content/gate_data"
OWN_REPEATS = 3
N_COMPOSITES = 400      # board-on-a-table pictures (pcb)
N_PLAIN = 300           # plain scenes (not_pcb)
SIZE = (640, 480)
rng_np = np.random.default_rng(5)

shutil.rmtree(DATA, ignore_errors=True)
for split in ("train", "val"):
    for label in ("pcb", "not_pcb"):
        os.makedirs(os.path.join(DATA, split, label))


def save_small(src, dst):
    img = ImageOps.exif_transpose(Image.open(src)).convert("RGB")   # respect phone rotation, drop transparency
    img.thumbnail((640, 640))
    img.save(dst, quality=90)


def split_list(items, val_fraction, seed):
    items = sorted(items)
    random.Random(seed).shuffle(items)
    n_val = max(1, round(val_fraction * len(items))) if items else 0
    return items[n_val:], items[:n_val]


def plain_surface():
    """A blank-looking picture: paper, a plain table or a wall, with lighting and texture."""
    w, h = SIZE
    base = rng_np.integers(60, 250, size=3).astype(np.float32)
    if rng_np.random() < 0.4:                                    # paper: near white or grey
        base = np.full(3, rng_np.integers(170, 250), np.float32) + rng_np.integers(-8, 8, size=3)
    img = np.ones((h, w, 3), np.float32) * base
    gx = np.linspace(-1, 1, w)[None, :, None]; gy = np.linspace(-1, 1, h)[:, None, None]
    img *= 1 + 0.25 * (rng_np.uniform(-1, 1) * gx + rng_np.uniform(-1, 1) * gy)     # uneven light
    if rng_np.random() < 0.5:                                    # wood-like grain
        freq = rng_np.uniform(0.02, 0.08)
        img *= (1 + 0.08 * np.sin(np.arange(h)[:, None, None] * freq + rng_np.normal(0, 2, (h, 1, 1))))
    img += rng_np.normal(0, rng_np.uniform(2, 10), img.shape)    # camera noise
    img = np.clip(img, 0, 255).astype(np.uint8)
    if rng_np.random() < 0.4:                                    # ruled lines or text-like scribbles
        for y in range(rng_np.integers(20, 60), h, rng_np.integers(18, 40)):
            if rng_np.random() < 0.5:
                cv2.line(img, (20, y), (w - 20, y), (110, 110, 140), 1)
            else:
                x = 30
                while x < w - 60:
                    n = int(rng_np.integers(15, 60))
                    cv2.line(img, (x, y), (x + n, y), (40, 40, 40), 2)
                    x += n + int(rng_np.integers(8, 20))
    if rng_np.random() < 0.3:                                    # a soft shadow
        mask = np.zeros((h, w), np.float32)
        cv2.circle(mask, (int(rng_np.integers(0, w)), int(rng_np.integers(0, h))), int(rng_np.integers(80, 300)), 1, -1)
        mask = cv2.GaussianBlur(mask, (0, 0), 40)[..., None]
        img = (img * (1 - 0.35 * mask)).astype(np.uint8)
    return img


def board_on_table(pcb_path, background_path):
    """A PCB photo shrunk, tilted and placed on a background, like a board lying under the camera."""
    w, h = SIZE
    if background_path and rng_np.random() < 0.6:
        bg = cv2.resize(cv2.imread(background_path), SIZE)
        bg = cv2.GaussianBlur(bg, (0, 0), 2)
    else:
        bg = plain_surface()
    board = cv2.imread(pcb_path)
    scale = rng_np.uniform(0.35, 0.9) * min(w / board.shape[1], h / board.shape[0])
    board = cv2.resize(board, None, fx=scale, fy=scale, interpolation=cv2.INTER_AREA)
    bh, bw = board.shape[:2]
    angle = rng_np.uniform(-25, 25)
    m = cv2.getRotationMatrix2D((bw / 2, bh / 2), angle, 1.0)
    cos, sin = abs(m[0, 0]), abs(m[0, 1])
    nw, nh = int(bh * sin + bw * cos), int(bh * cos + bw * sin)
    m[0, 2] += nw / 2 - bw / 2; m[1, 2] += nh / 2 - bh / 2
    rot = cv2.warpAffine(board, m, (nw, nh))
    mask = cv2.warpAffine(np.full((bh, bw), 255, np.uint8), m, (nw, nh))
    if nw > w or nh > h:
        f = min(w / nw, h / nh) * 0.95
        rot = cv2.resize(rot, None, fx=f, fy=f); mask = cv2.resize(mask, None, fx=f, fy=f)
        nh, nw = rot.shape[:2]
    x = int(rng_np.integers(0, w - nw + 1)); y = int(rng_np.integers(0, h - nh + 1))
    roi = bg[y:y + nh, x:x + nw]
    roi[mask > 0] = rot[mask > 0]
    out = bg.astype(np.float32) * rng_np.uniform(0.7, 1.2) + rng_np.uniform(-15, 15, 3)   # lighting and colour cast
    out += rng_np.normal(0, 3, out.shape)
    return np.clip(out, 0, 255).astype(np.uint8)


# Generated pictures go into a staging folder first, then are split like the others.
GEN = "/content/gate_generated"
shutil.rmtree(GEN, ignore_errors=True)
os.makedirs(f"{GEN}/pcb"); os.makedirs(f"{GEN}/not_pcb")
for i in range(N_COMPOSITES):
    cv2.imwrite(f"{GEN}/pcb/composite_{i:04d}.jpg",
                board_on_table(random.Random(i).choice(dataset_pcb), random.Random(i + 1).choice(everyday)))
for i in range(N_PLAIN):
    cv2.imwrite(f"{GEN}/not_pcb/plain_{i:04d}.jpg", plain_surface())
generated = {label: sorted(glob.glob(f"{GEN}/{label}/*.jpg")) for label in ("pcb", "not_pcb")}

own_val = {}                       # your own test photos, scored separately in Step 8 (if used)
for label, others in (("pcb", dataset_pcb + generated["pcb"]), ("not_pcb", everyday + generated["not_pcb"])):
    own_train, own_val[label] = split_list(own[label], 0.25, seed=1)
    other_train, other_val = split_list(others, 0.15, seed=2)
    n = {"train": 0, "val": 0}
    for i, src in enumerate(other_train):
        save_small(src, f"{DATA}/train/{label}/other_{i:05d}.jpg"); n["train"] += 1
    for i, src in enumerate(other_val):
        save_small(src, f"{DATA}/val/{label}/other_{i:05d}.jpg"); n["val"] += 1
    for i, src in enumerate(own_train):
        for r in range(OWN_REPEATS):
            save_small(src, f"{DATA}/train/{label}/own_{i:04d}_r{r}.jpg"); n["train"] += 1
    for i, src in enumerate(own_val[label]):
        save_small(src, f"{DATA}/val/{label}/own_{i:04d}.jpg"); n["val"] += 1
    print(f"{label:8s}: train {n['train']:5d} pictures, val {n['val']:4d} pictures "
          f"(generated: {len(generated[label])}, your own: {len(own[label])})")

## Step 6: Look at a few training pictures
Top row should all show a PCB (some small, on a background); bottom row should show no PCB.

In [ ]:
from IPython.display import display

sheet = Image.new("RGB", (6 * 200, 2 * 200), "white")
for row, label in enumerate(("pcb", "not_pcb")):
    files = sorted(glob.glob(f"{DATA}/train/{label}/*.jpg"))
    own_files = [f for f in files if "/own_" in f]
    picks = random.Random(3).sample(own_files, min(2, len(own_files)))
    picks += random.Random(4).sample(files, 6 - len(picks))
    for col, path in enumerate(picks):
        tile = Image.open(path).convert("RGB"); tile.thumbnail((196, 196))
        sheet.paste(tile, (col * 200 + 2, row * 200 + 2))
display(sheet)

## Step 7: Train
- `yolov8n-cls.pt` is the smallest YOLO classifier, already trained on ImageNet. We fine-tune it for our two classes.
- `epochs=20`, `patience=8`: up to 20 passes, stopping early if it stops improving.

This takes roughly 5 to 15 minutes. The `top1_acc` column (accuracy on the test pictures) should get close to 1.0.

In [ ]:
from ultralytics import YOLO

model = YOLO("yolov8n-cls.pt")
model.train(data=DATA, epochs=20, patience=8, imgsz=224, batch=64, device=0,
            project="/content/runs", name="pcb_gate", exist_ok=True)
save_dir = str(model.trainer.save_dir)
best = YOLO(os.path.join(save_dir, "weights", "best.pt"))

## Step 8: Score it on the test pictures
The Pi treats a picture as a PCB when the probability of `pcb` is at least 0.60, so the same rule is used here.
Every test picture is scored, and the counts are given for each kind: dataset photos, composites, everyday photos,
plain scenes and, if used, your own photos. Mistakes are listed by file name.

Remember that all of these except your own photos are downloaded or generated, so a high score here does not
prove the check works at the station. Test it there too (see the instructions).

In [ ]:
THRESHOLD = 0.60


def p_pcb(path):
    r = best(path, imgsz=224, verbose=False)[0]
    pcb_index = [i for i, name in r.names.items() if name == "pcb"][0]
    return float(r.probs.data[pcb_index])


def kind_of(path):
    name = os.path.basename(path)
    if name.startswith("own_"):
        return "your own photos"
    src_kind = {"pcb": "PCB dataset photos + composites", "not_pcb": "everyday photos + plain scenes"}
    return src_kind[os.path.basename(os.path.dirname(path))]


tally = {}
mistakes = []
for label in ("pcb", "not_pcb"):
    for path in sorted(glob.glob(f"{DATA}/val/{label}/*.jpg")):
        p = p_pcb(path)
        said = "pcb" if p >= THRESHOLD else "not_pcb"
        key = (label, kind_of(path))
        right, total = tally.get(key, (0, 0))
        tally[key] = (right + (said == label), total + 1)
        if said != label:
            mistakes.append(f"{os.path.basename(path)}: is {label}, model said {said} (p_pcb = {p:.2f})")

lines = ["PCB check (YOLOv8n-cls, 224 x 224), threshold p(pcb) >= 0.60",
         "Test pictures recognised correctly:"]
for (label, kind), (right, total) in sorted(tally.items()):
    lines.append(f"  {label:8s} {kind:34s} {right}/{total}")
all_right = sum(r for r, t in tally.values()); all_total = sum(t for r, t in tally.values())
lines.append(f"Overall: {all_right}/{all_total} = {all_right / all_total:.3f}")
if not USE_OWN_PHOTOS:
    lines.append("No photos from the station camera were used; test the check at the station before relying on it.")
summary = "\n".join(lines) + "\n"
print(summary)
print(f"Mistakes ({len(mistakes)}):" if mistakes else "No mistakes on the test pictures.")
for m in mistakes[:40]:
    print("  " + m)

## Step 9: Convert for the Raspberry Pi and download
The zip holds `pcb_gate.onnx` for the Pi, `pcb_gate.pt` (keep it for re-training) and `gate_metrics.txt`.

In [ ]:
from google.colab import files

onnx_path = best.export(format="onnx", imgsz=224)
out = "/content/pcb_gate"
shutil.rmtree(out, ignore_errors=True)
os.makedirs(out)
shutil.copy(onnx_path, os.path.join(out, "pcb_gate.onnx"))
shutil.copy(os.path.join(save_dir, "weights", "best.pt"), os.path.join(out, "pcb_gate.pt"))
with open(os.path.join(out, "gate_metrics.txt"), "w") as f:
    f.write(summary + "\n" + "\n".join(mistakes) + "\n")
shutil.make_archive(out, "zip", out)
files.download(out + ".zip")